In [1]:
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [2]:
from google.colab import files
import os

os.makedirs("data", exist_ok=True)

print("Upload train.csv, dev.csv, and test.csv")
uploaded = files.upload()

for filename in uploaded.keys():
    new_path = os.path.join("data", filename)
    if os.path.exists(new_path):
        os.remove(new_path)
    os.rename(filename, new_path)
    print(f"Saved: {new_path}")

print("Files in data/:", os.listdir("data"))

Upload train.csv, dev.csv, and test.csv


Saving train.csv to train.csv
Saving dev.csv to dev.csv
Saving test.csv to test.csv
Saved: data/train.csv
Saved: data/dev.csv
Saved: data/test.csv
Files in data/: ['test.csv', 'train.csv', 'dev.csv']


## Solution C – Transformer-based NLI

This solution models Natural Language Inference as a binary sequence-pair classification task using a pretrained transformer.

Given a premise and a hypothesis, the model predicts whether the hypothesis is entailed by the premise. Compared with traditional TF-IDF-based approaches, transformer models are better able to capture contextual meaning and semantic relationships between sentence pairs.

In [3]:
import pandas as pd

train_df = pd.read_csv("data/train.csv")
dev_df = pd.read_csv("data/dev.csv")
test_df = pd.read_csv("data/test.csv")

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Test:", test_df.shape)

print("\nTrain label distribution:")
print(train_df["label"].value_counts())

print("\nDev label distribution:")
print(dev_df["label"].value_counts())

display(train_df.head(3))
display(dev_df.head(3))
display(test_df.head(3))

Train: (24432, 3)
Dev: (6736, 3)
Test: (3302, 2)

Train label distribution:
label
1    12648
0    11784
Name: count, dtype: int64

Dev label distribution:
label
1    3478
0    3258
Name: count, dtype: int64


,premise,hypothesis,label
0,yeah i don't know cut California in half or so...,Yeah. I'm not sure how to make that fit. Maybe...,1
1,actual names will not be used,"For the sake of privacy, actual names are not ...",1
2,The film was directed by Randall Wallace.,The film was directed by Randall Wallace and s...,1


,premise,hypothesis,label
0,"By starting at the soft underbelly, the 16,000...","General Nelson A. Miles had 30,000 troops in h...",0
1,"The class had broken into a light sweat, but w...",The class grew more tense as time went on.,1
2,"Samson had his famous haircut here, but he wou...",It was unknown where exactly within the town S...,1


,premise,hypothesis
0,"Boy wearing red hat, blue jacket pushing plow ...",The boy is surrounded by snow
1,A blond woman in a black shirt is standing beh...,The woman is standing.
2,Three people in uniform are outdoors and are o...,Uniformed people are outside


In [4]:
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train_df[["premise", "hypothesis", "label"]],
    preserve_index=False
)
dev_ds = Dataset.from_pandas(
    dev_df[["premise", "hypothesis", "label"]],
    preserve_index=False
)
test_ds = Dataset.from_pandas(
    test_df[["premise", "hypothesis"]],
    preserve_index=False
)

print(train_ds)
print(dev_ds)
print(test_ds)

Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 24432
})
Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 6736
})
Dataset({
    features: ['premise', 'hypothesis'],
    num_rows: 3302
})


In [5]:
from transformers import AutoTokenizer

MODEL_NAME = "distilroberta-base"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

train_tok = train_ds.map(tokenize_batch, batched=True)
dev_tok = dev_ds.map(tokenize_batch, batched=True)
test_tok = test_ds.map(tokenize_batch, batched=True)

train_tok = train_tok.remove_columns(["premise", "hypothesis"])
dev_tok = dev_tok.remove_columns(["premise", "hypothesis"])
test_tok = test_tok.remove_columns(["premise", "hypothesis"])

train_tok.set_format("torch")
dev_tok.set_format("torch")
test_tok.set_format("torch")

print(train_tok)
print(dev_tok)
print(test_tok)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/24432 [00:00<?, ? examples/s]

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

Map:   0%|          | 0/3302 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 24432
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 6736
})
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 3302
})


## Model Training

We fine-tune a pretrained transformer for binary classification. This is a Category C approach under the coursework rubric, since it is a deep learning solution based on a transformer architecture.

The model is selected to provide strong semantic performance while remaining feasible to train on free Colab.

In [6]:
import numpy as np
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print(model.__class__.__name__)

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification


In [7]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, matthews_corrcoef

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    mcc = matthews_corrcoef(labels, preds)

    return {
        "accuracy": acc,
        "macro_precision": p,
        "macro_recall": r,
        "macro_f1": f1,
        "mcc": mcc,
    }

In [8]:
import torch
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="nli_distilroberta_outputs",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    fp16=torch.cuda.is_available()
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [9]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=dev_tok,
    compute_metrics=compute_metrics,
)

train_output = trainer.train()
print(train_output)

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Mcc
1,0.513041,0.440062,0.805819,0.812662,0.808175,0.805413,0.620821
2,0.323945,0.418866,0.832838,0.832862,0.832369,0.832549,0.665231


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=3054, training_loss=0.418492954856208, metrics={'train_runtime': 371.9273, 'train_samples_per_second': 131.381, 'train_steps_per_second': 8.211, 'total_flos': 3236443483963392.0, 'train_loss': 0.418492954856208, 'epoch': 2.0})


In [10]:
eval_results = trainer.evaluate()
print(eval_results)

{'eval_loss': 0.418866366147995, 'eval_accuracy': 0.8328384798099763, 'eval_macro_precision': 0.8328622573958923, 'eval_macro_recall': 0.8323687505537747, 'eval_macro_f1': 0.8325491459422187, 'eval_mcc': 0.6652308248936804, 'eval_runtime': 11.5578, 'eval_samples_per_second': 582.808, 'eval_steps_per_second': 36.425, 'epoch': 2.0}


In [11]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

dev_output = trainer.predict(dev_tok)
dev_logits = dev_output.predictions
dev_pred = np.argmax(dev_logits, axis=-1)
y_dev = dev_df["label"].values

print("Confusion matrix:")
print(confusion_matrix(y_dev, dev_pred))

print("\nClassification report:")
print(classification_report(y_dev, dev_pred, digits=4))

Confusion matrix:
[[2665  593]
 [ 533 2945]]

Classification report:
              precision    recall  f1-score   support

           0     0.8333    0.8180    0.8256      3258
           1     0.8324    0.8468    0.8395      3478

    accuracy                         0.8328      6736
   macro avg     0.8329    0.8324    0.8325      6736
weighted avg     0.8328    0.8328    0.8328      6736



## Error Analysis

The transformer-based model substantially outperforms a traditional lexical approach because it captures contextual meaning rather than relying mainly on surface overlap.

Remaining errors are likely to come from subtle semantic distinctions, implicit reasoning, ambiguous phrasing, or cases where world knowledge is helpful. This highlights that even strong contextual models can still struggle on difficult NLI examples.

In [13]:
from datasets import Dataset

full_df = pd.concat([train_df, dev_df], ignore_index=True)
print("Full labeled data shape:", full_df.shape)

full_ds = Dataset.from_pandas(
    full_df[["premise", "hypothesis", "label"]],
    preserve_index=False
)

full_tok = full_ds.map(tokenize_batch, batched=True)
full_tok = full_tok.remove_columns(["premise", "hypothesis"])
full_tok.set_format("torch")

final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Full labeled data shape: (31168, 3)


Map:   0%|          | 0/31168 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Final Training and Test Prediction

After selecting the best configuration using development data, we retrain the model on the combined training and development sets to maximise the use of available labelled data, then generate predictions for the released test set.

In [16]:
import torch
from transformers import TrainingArguments

final_training_args = TrainingArguments(
    output_dir="nli_distilroberta_final",
    eval_strategy="no",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=16,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    save_total_limit=1,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [17]:
from transformers import Trainer

final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=full_tok,
)

final_train_output = final_trainer.train()
print(final_train_output)

Step,Training Loss
1948,0.498177
3896,0.308999


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3896, training_loss=0.40358795228680056, metrics={'train_runtime': 464.042, 'train_samples_per_second': 134.333, 'train_steps_per_second': 8.396, 'total_flos': 4128743881310208.0, 'train_loss': 0.40358795228680056, 'epoch': 2.0})


In [18]:
test_output = final_trainer.predict(test_tok)
test_logits = test_output.predictions
test_pred = np.argmax(test_logits, axis=-1)

print("Test predictions generated:", len(test_pred))
print(pd.Series(test_pred).value_counts())

Test predictions generated: 3302
1    1718
0    1584
Name: count, dtype: int64


In [19]:
GROUP_NUMBER = "56"

pred_df = pd.DataFrame({"prediction": test_pred})
pred_filename = f"Group_{GROUP_NUMBER}_C.csv"
pred_df.to_csv(pred_filename, index=False)

print("Saved:", pred_filename)
display(pred_df.head())

Saved: Group_56_C.csv


,prediction
0,1
1,1
2,1
3,1
4,1


In [20]:
SAVE_DIR = "solution_C_model"

final_model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved model and tokenizer to", SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model and tokenizer to solution_C_model


In [21]:
import shutil
import os

zip_path = shutil.make_archive("solution_C_model", "zip", SAVE_DIR)
print("Created:", zip_path)
print("Zip size (MB):", round(os.path.getsize("solution_C_model.zip") / (1024 * 1024), 2))

Created: /content/solution_C_model.zip
Zip size (MB): 291.52


In [23]:
from google.colab import files

files.download(pred_filename)
files.download("solution_C_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>